# 0. 분석 패키지 설치

In [7]:
import sys
print(sys.version)

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]


In [8]:
# conda install -c conda-forge jpype1

In [9]:
# !pip install konlpy

# 1.라이브러리 불러오기

In [10]:
import numpy as np  
import pandas as pd
import re
import string
import konlpy
from konlpy.tag import Okt

# 2.형태소

- Okt, 코모란, 한나눔, 꼬고마, 메캅 등 5개 오픈소스 형태소 분석기를 파이썬 환경에서 사용할 수 있도록 인터페이스를 통일한 한국어 자연처리 패키지

    - Otk
    - Komoran
    - Hannanum
    - Kkma
    - Mecab

In [11]:
import os
# os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.11"

from konlpy.tag import Kkma

In [12]:
tokenizer = Kkma()

In [13]:
# morphs() 함수 : 입력된 문장을 단어(형태소) 단위로 나누어 리스트로 반환
sentence = '아버지가방에들어가신다.'
tokenizer.morphs(sentence)

['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']

In [14]:
# pos() 함수 : 입력된 문장을 형태소 단위로 나누고 각 형태소의 품사 태그를 함께 반환
tokenizer.pos(sentence)

[('아버지', 'NNG'),
 ('가방', 'NNG'),
 ('에', 'JKM'),
 ('들어가', 'VV'),
 ('시', 'EPH'),
 ('ㄴ다', 'EFN'),
 ('.', 'SF')]

In [15]:
from konlpy.tag import Okt, Komoran, Hannanum, Kkma

In [16]:
# 형태소 분석기 함수 만들기
def get_tokenizer(tokenizer_name):
    if tokenizer_name == 'Okt':
        tokenizer = Okt()
    elif tokenizer_name == 'Komoran':
        tokenizer = Komoran()
    elif tokenizer_name == 'Hannanum':
        tokenizer = Hannanum()
    else:
        tokenizer = Kkma()
    return tokenizer

In [17]:
tokenizer = get_tokenizer('Okt')
print('Okt 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Okt 형태소 분석기
['아버지', '가방', '에', '들어가신다', '.']
[('아버지', 'Noun'), ('가방', 'Noun'), ('에', 'Josa'), ('들어가신다', 'Verb'), ('.', 'Punctuation')]


In [18]:
tokenizer = get_tokenizer('Komoran')
print('Komoran 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Komoran 형태소 분석기
['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']
[('아버지', 'NNG'), ('가방', 'NNP'), ('에', 'JKB'), ('들어가', 'VV'), ('시', 'EP'), ('ㄴ다', 'EF'), ('.', 'SF')]


In [19]:
tokenizer = get_tokenizer('Hannanum')
print('Hannanum 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Hannanum 형태소 분석기
['아버지가방에들어가', '이', '시ㄴ다', '.']
[('아버지가방에들어가', 'N'), ('이', 'J'), ('시ㄴ다', 'E'), ('.', 'S')]


In [20]:
tokenizer = get_tokenizer('Kkma')
print('Kkma 형태소 분석기')
print(tokenizer.morphs(sentence))
print(tokenizer.pos(sentence))

Kkma 형태소 분석기
['아버지', '가방', '에', '들어가', '시', 'ㄴ다', '.']
[('아버지', 'NNG'), ('가방', 'NNG'), ('에', 'JKM'), ('들어가', 'VV'), ('시', 'EPH'), ('ㄴ다', 'EFN'), ('.', 'SF')]


# 3.텍스트 전처리

## 3.1.데이터 수집

In [21]:
# 예제 데이터
reviews = [
    "이 영화 정말 재미있었어요! 강력 추천합니다. 👍",
    "완전 지루하고 시간 낭비였습니다. 😡",
    "연기는 훌륭했지만, 스토리는 조금 아쉬웠어요. 🤔",
    "명작입니다. 꼭 보세요! 👏",
    "별로였습니다. 기대보다 실망이 컸어요. 😞",
    '친구들과 함께하면 더욱 재미있어요!',
    '그래픽이 아주 멋집니다. 추천해요!',
    '초반에는 재미있는데, 시간이 지나면 지루해집니다.'
]

## 3.2.텍스트 정제 (특수 문자 제거, 소문자 변환)

In [22]:
reviews[0]

'이 영화 정말 재미있었어요! 강력 추천합니다. 👍'

In [23]:
# 한글과 공백만 남기기

re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', reviews[0])

'이 영화 정말 재미있었어요 강력 추천합니다 '

In [24]:
def clean_text(text):
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text)
    text = text.lower()         # 소문자 변환
    return text

In [25]:
# reviews 리스트를 for문 돌려서 텍스트 클리닝 작업
cleaned_reviews = [clean_text(review) for review in reviews]
cleaned_reviews

['이 영화 정말 재미있었어요 강력 추천합니다 ',
 '완전 지루하고 시간 낭비였습니다 ',
 '연기는 훌륭했지만 스토리는 조금 아쉬웠어요 ',
 '명작입니다 꼭 보세요 ',
 '별로였습니다 기대보다 실망이 컸어요 ',
 '친구들과 함께하면 더욱 재미있어요',
 '그래픽이 아주 멋집니다 추천해요',
 '초반에는 재미있는데 시간이 지나면 지루해집니다']

## 3.3.토큰화 (Tokenization)

In [26]:
# Okt 형태소 분석기 초기화
okt = Okt()

In [27]:
# Okt 형태소 분석기를 사용하여 토큰화
okt_tokenized_reviews = [okt.morphs(review) for review in cleaned_reviews]
print('Okt 형태소 단위 토큰화')
print(okt_tokenized_reviews)

Okt 형태소 단위 토큰화
[['이', '영화', '정말', '재미있었어요', '강력', '추천', '합니다'], ['완전', '지루하고', '시간', '낭비', '였습니다'], ['연기', '는', '훌륭했지만', '스토리', '는', '조금', '아쉬웠어요'], ['명작', '입니다', '꼭', '보세요'], ['별로', '였습니다', '기대', '보다', '실망', '이', '컸어요'], ['친구', '들', '과', '함께', '하면', '더욱', '재미있어요'], ['그래픽', '이', '아주', '멋집니다', '추천', '해요'], ['초반', '에는', '재미있는데', '시간', '이', '지나면', '지루해', '집니다']]


## 3.4.불용어 제거

In [28]:
# 불용어 목록
stop_words = ['이', '정말', '완전', '조금', '는', '꼭']

In [29]:
reviews_no_stopwords = []

for tokens in okt_tokenized_reviews:
    filtered_tokens = []
    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)
    reviews_no_stopwords.append(filtered_tokens)

print(reviews_no_stopwords)


[['영화', '재미있었어요', '강력', '추천', '합니다'], ['지루하고', '시간', '낭비', '였습니다'], ['연기', '훌륭했지만', '스토리', '아쉬웠어요'], ['명작', '입니다', '보세요'], ['별로', '였습니다', '기대', '보다', '실망', '컸어요'], ['친구', '들', '과', '함께', '하면', '더욱', '재미있어요'], ['그래픽', '아주', '멋집니다', '추천', '해요'], ['초반', '에는', '재미있는데', '시간', '지나면', '지루해', '집니다']]


In [30]:
# 불용어 처리하는 함수
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

reviews_no_stopwords = [remove_stopwords(tokens) for tokens in okt_tokenized_reviews]
print(reviews_no_stopwords)

[['영화', '재미있었어요', '강력', '추천', '합니다'], ['지루하고', '시간', '낭비', '였습니다'], ['연기', '훌륭했지만', '스토리', '아쉬웠어요'], ['명작', '입니다', '보세요'], ['별로', '였습니다', '기대', '보다', '실망', '컸어요'], ['친구', '들', '과', '함께', '하면', '더욱', '재미있어요'], ['그래픽', '아주', '멋집니다', '추천', '해요'], ['초반', '에는', '재미있는데', '시간', '지나면', '지루해', '집니다']]


## 3.5.어간 추출 (Stemming) 및 표제어 추출 (Lemmatization)
- 한글은 영어와 달리 어미, 조사, 접사가 많고, 복잡한 문법 구조를 가진다.

In [31]:
# 형태소 추출 결과를 저장할 리스트
stemmed_reviews = []

for tokens in reviews_no_stopwords:
    stemmed_tokens = []
    for token in tokens:        # Okt 형태소 분석기의 morphs 메소드를 사용해서 각 토큰의 형태소 추출
        stemmed = okt.morphs(token, stem=True)      # 어간 추출 옵션 사용.
        stemmed_tokens.append(stemmed)

    stemmed_reviews.append(stemmed_tokens)

stemmed_reviews                          


[[['영화'], ['재미있다'], ['강력'], ['추천'], ['하다']],
 [['지루하다'], ['시간'], ['낭비'], ['이다']],
 [['연기'], ['훌륭하다'], ['스토리'], ['아쉽다']],
 [['명작'], ['이다'], ['보다']],
 [['별로'], ['이다'], ['기대'], ['보다'], ['실망'], ['크다']],
 [['친구'], ['들다'], ['과'], ['함께'], ['하다'], ['더욱'], ['재미있다']],
 [['그래픽'], ['아주'], ['멋지다'], ['추천'], ['해', '요']],
 [['초반'], ['에는'], ['재미있다'], ['시간'], ['지나다'], ['지루하다'], ['지다']]]

In [32]:
def stem_tokens(tokens):
    return [okt.morphs(token, stem=True) for token in tokens]

stemmed_reviews = [stem_tokens(tokens) for tokens in reviews_no_stopwords]
stemmed_reviews

[[['영화'], ['재미있다'], ['강력'], ['추천'], ['하다']],
 [['지루하다'], ['시간'], ['낭비'], ['이다']],
 [['연기'], ['훌륭하다'], ['스토리'], ['아쉽다']],
 [['명작'], ['이다'], ['보다']],
 [['별로'], ['이다'], ['기대'], ['보다'], ['실망'], ['크다']],
 [['친구'], ['들다'], ['과'], ['함께'], ['하다'], ['더욱'], ['재미있다']],
 [['그래픽'], ['아주'], ['멋지다'], ['추천'], ['해', '요']],
 [['초반'], ['에는'], ['재미있다'], ['시간'], ['지나다'], ['지루하다'], ['지다']]]

In [33]:
# 형용사만 추출

text = "이 카페는 아늑하고 조용하며 분위기가 정말 독특하고 예뻐요. 커피도 향긋하고 맛있어요!"

In [34]:
okt.pos(text)

[('이', 'Noun'),
 ('카페', 'Noun'),
 ('는', 'Josa'),
 ('아늑하고', 'Adjective'),
 ('조용하며', 'Adjective'),
 ('분위기', 'Noun'),
 ('가', 'Josa'),
 ('정말', 'Noun'),
 ('독특하고', 'Adjective'),
 ('예뻐요', 'Adjective'),
 ('.', 'Punctuation'),
 ('커피', 'Noun'),
 ('도', 'Josa'),
 ('향', 'Noun'),
 ('긋하고', 'Verb'),
 ('맛있어요', 'Adjective'),
 ('!', 'Punctuation')]

In [35]:
# 형용사를 저장할 리스트
adjectives = []

# 품사 태깅 및 형용사 추출
for word, pos in okt.pos(text):
    if pos == 'Adjective':
        adjectives.append(word)

print("형용사 추출 결과 : ", adjectives)

형용사 추출 결과 :  ['아늑하고', '조용하며', '독특하고', '예뻐요', '맛있어요']


In [36]:
# 명사만 추출
def extract_nouns(tokens):
    nouns_list = []
    for token in tokens:
        nouns = okt.nouns(token)
        nouns_list.extend(nouns)
    return nouns_list

In [37]:
noun_reviews = [extract_nouns(tokens) for tokens in reviews_no_stopwords]
noun_reviews

[['영화', '강력', '추천'],
 ['시간', '낭비'],
 ['연기', '스토리'],
 ['명작'],
 ['별로', '기대', '실망'],
 ['친구', '과', '더욱'],
 ['그래픽', '아주', '추천', '해'],
 ['초반', '시간']]

## 3.6.데이터 분석(EDA)
- 빈도 분석
- 트렌드 분석
- 감성 분석
- 트렌드, 빈도, 감성 분석에 사용할 데이터 형태로 전처리 했다면 알맞은 시각화 방법을 통해 분석해봅시다.

    - 예시 : 시간대별 리뷰 추이가 궁금하다면 -> 시간대별로 데이터 프레임 만든 후, 상위 단어 추출하여 분석
    - 예시 : 전체 리뷰의 상태가 궁금하다면 -> 전체 리뷰를 합쳐서 단어 분석